# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields with respective `@id`s.


In [ ]:
# List all record sets by their @id and examine their fields
print("Available record sets and fields:")
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
for rs in metadata.record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '(no name)')}")
    print("  Fields:")
    for field in rs['fields']:
        print(f"    - Field @id: {field['@id']}, name: {field.get('name', '(no name)')}, dataType: {field.get('dataType', '(no dataType)')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For this dataset, let's select the main record set which contains the protein measurements.
# You can inspect the printed record sets above and choose the appropriate @id. Example below uses the first one.

if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Selected RecordSet: {main_record_set_id}\n")

    # Load all records for all record sets into pandas DataFrames
    dataframes = {}
    for record_set_id in record_set_ids:
        try:
            records = list(dataset.records(record_set=record_set_id))
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for RecordSet {record_set_id}")
        except Exception as e:
            print(f"Could not load records for {record_set_id}: {e}")

    main_df = dataframes[main_record_set_id]
    print("\nAvailable columns in the main record set:")
    print(main_df.columns.tolist())
    main_df.head()
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Let's select a numeric field (e.g., 'MW' for molecular weight, or similar fields as found in step 2/3), filter records, normalize, and group data.


In [ ]:
# Example numeric field for analysis: Replace with a field appropriate for the dataset as displayed above
# We'll attempt common field names e.g. 'mw', 'MW', 'molecular_weight', 'MolecularWeight', or select the first numeric field
import numpy as np

numeric_field_candidates = ['MW', 'mw', 'MolecularWeight', 'molecular_weight', 'coverage', 'peptide_count', 'normalized_abundance', 'pI']
main_df_numeric = None
numeric_field = None

# Try to find the first available numeric field
for candidate in numeric_field_candidates:
    if candidate in main_df.columns:
        numeric_field = candidate
        main_df_numeric = pd.to_numeric(main_df[numeric_field], errors='coerce')
        break
if numeric_field is None:
    # Otherwise, pick first column with numeric dtype
    for col in main_df.columns:
        if pd.api.types.is_numeric_dtype(main_df[col]):
            numeric_field = col
            main_df_numeric = main_df[col]
            break
if numeric_field is None:
    print("No numeric field found for analysis.")
else:
    threshold = main_df_numeric.mean()
    filtered_df = main_df[main_df_numeric > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.2f} (mean): {len(filtered_df)} records")

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field].astype(float) - filtered_df[numeric_field].astype(float).mean()) / filtered_df[numeric_field].astype(float).std()
    print(f"\nNormalized example:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"].replace(numeric_field, str(numeric_field))].head())

    # Attempt to group by a possible categorical field
    group_field_candidates = ['description', 'ProteinID', 'accession', 'sample']
    group_field = None
    for candidate in group_field_candidates:
        if candidate in filtered_df.columns:
            group_field = candidate
            break

    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped means by '{group_field}' (showing first 5):\n", grouped_df.head())
    else:
        print("No suitable group field found for grouping analysis.")

## 5. Visualization
Visualize distribution of the selected numeric field and any relationships found in the data.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(main_df[numeric_field].astype(float), bins=30, kde=True, color='skyblue')
    plt.title(f"Distribution of '{numeric_field}' in main record set")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()

    if group_field is not None:
        # Visualize group means if grouping field is available
        plt.figure(figsize=(10, 6))
        top_groups = grouped_df.sort_values(numeric_field, ascending=False).head(10)
        sns.barplot(x=top_groups[numeric_field], y=top_groups[group_field], palette="Blues_d")
        plt.title(f"Top 10 {group_field} by mean {numeric_field}")
        plt.xlabel(f"Mean {numeric_field}")
        plt.ylabel(group_field)
        plt.show()

## 6. Conclusion
In this notebook, we've explored the protein measurement dataset using the `mlcroissant` library. We loaded the dataset, reviewed its structure, performed basic filtering and normalization on numeric fields, grouped data by key attributes, and visualized important distributions. This workflow can be adapted for further in-depth analyses or integration into ML pipelines.
